# 06 — Alignement des entités inter-sources

C'est ici que les entités décrivant **le même concept** dans des sources différentes seront rapprochées.

In [1]:
import sys, os
import csv
import numpy as np
import jobkb_common as C
from collections import defaultdict, Counter

def load(path):
    with open(path, encoding="utf-8") as f:
        return list(csv.DictReader(f))

occs   = load(C.OCCUPATIONS_CSV)
labels = load(C.LABELS_CSV)
id2occ = {o["entity_id"]: o for o in occs}

# On aligne les VRAIES professions (pas les nœuds de groupes ISCO)
REAL_SRC = {"ESCO", "ROME", "WIKIDATA", "REMOTEOK", "ARBEITNOW"}
by_src = defaultdict(list)
for o in occs:
    if o["source"] in REAL_SRC:
        by_src[o["source"]].append(o)
print("Professions à aligner par source :", {k: len(v) for k, v in by_src.items()})
if len(by_src) < 2:
    print("\n⚠ Une seule source présente — aucun alignement inter-sources possible.")
    print("  Exécutez 03 (ROME/Wikidata) pour avoir de quoi réconcilier.")

Professions à aligner par source : {'ESCO': 83, 'ROME': 96, 'REMOTEOK': 19, 'ARBEITNOW': 58, 'WIKIDATA': 1}


## 6.1. Libellé exact

Deux entités de sources différentes qui partagent un `label_norm` identique sont candidates. On pondère par le type de libellé.

In [6]:
TYPE_W = {"preferred": 1.0, "alt": 0.7, "hidden": 0.5}

lab_idx = defaultdict(list)
for l in labels:
    o = id2occ.get(l["entity_id"])
    if l["label_norm"] and l["entity_kind"] == "occupation" and o and o["source"] in REAL_SRC:
        lab_idx[l["label_norm"]].append(l)

exact_pairs = {}
for norm, rows in lab_idx.items():
    if len({r["source"] for r in rows}) < 2:
        continue
    for i in range(len(rows)):
        for j in range(i + 1, len(rows)):
            a, b = rows[i], rows[j]
            if a["source"] == b["source"]:
                continue
            key = tuple(sorted([a["entity_id"], b["entity_id"]]))
            w = min(TYPE_W.get(a["label_type"], .5), TYPE_W.get(b["label_type"], .5))
            if key not in exact_pairs or w > exact_pairs[key]["w"]:
                exact_pairs[key] = {"w": w, "norm": norm, "ta": a["label_type"], "tb": b["label_type"]}

print(f"paires candidates (libellé exact partagé) : {len(exact_pairs)}")

paires candidates (libellé exact partagé) : 15


## 6.2. Quasi-Similarité

On va utiliser un modèle multilingue (`paraphrase-multilingual-MiniLM-L12-v2`), utile pour rapprocher des libellés sémantiquement proches mais orthographiquement différent. Sinon TF-IDF sur n-grammes de caractères (3–5) (en utilisant scikit-learn).

In [11]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

USE_EMBEDDINGS = True
FUZZY_THRESHOLD = 0.5  

def repr_text(o):
    parts = [o["pref_label_fr"] or o["pref_label_en"]]
    if o["alt_labels_fr"]:
        parts.append(o["alt_labels_fr"].replace(" | ", " "))
    return C.normalize_label(" ".join(p for p in parts if p))

def similarity_matrix(A, B):
    """Retourne une matrice de similarité |A|x|B| dans [0,1]."""
    if USE_EMBEDDINGS:
        model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
        # ici on garde les libellés lisibles
        ea = model.encode([ (o["pref_label_fr"] or o["pref_label_en"]) for o in A ], normalize_embeddings=True)
        eb = model.encode([ (o["pref_label_fr"] or o["pref_label_en"]) for o in B ], normalize_embeddings=True)
        return cosine_similarity(ea, eb)
    else:

        txtA = [repr_text(o) for o in A]
        txtB = [repr_text(o) for o in B]
        vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=1)
        X = vec.fit_transform(txtA + txtB)
        return cosine_similarity(X[:len(A)], X[len(A):])

fuzzy_pairs = {}
srcs = sorted(by_src)
for x in range(len(srcs)):
    for y in range(x + 1, len(srcs)):
        A, B = by_src[srcs[x]], by_src[srcs[y]]
        sim = similarity_matrix(A, B)
        # meilleur B pour chaque A, et meilleur A pour chaque B (symétrie)
        for i, a in enumerate(A):
            j = int(np.argmax(sim[i])); s = float(sim[i][j])
            if s >= FUZZY_THRESHOLD:
                key = tuple(sorted([a["entity_id"], B[j]["entity_id"]]))
                fuzzy_pairs[key] = max(fuzzy_pairs.get(key, 0), s)
        for j, b in enumerate(B):
            i = int(np.argmax(sim[:, j])); s = float(sim[i][j])
            if s >= FUZZY_THRESHOLD:
                key = tuple(sorted([b["entity_id"], A[i]["entity_id"]]))
                fuzzy_pairs[key] = max(fuzzy_pairs.get(key, 0), s)

print(f"\npaires floues retenues (seuil {FUZZY_THRESHOLD}) : {len(fuzzy_pairs)}")
print("Méthode :", "embeddings multilingues" if USE_EMBEDDINGS else "TF-IDF n-grammes de caractères")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3588.73it/s]



paires floues retenues (seuil 0.5) : 534
Méthode : embeddings multilingues


## 6.3. Combinaison → confiance + relation SKOS

On fusionne les deux signaux. La relation SKOS est dérivée de la confiance :
- `≥ 0.90` → `skos:exactMatch`
- `≥ 0.70` → `skos:closeMatch`
- sinon → `skos:relatedMatch`

In [12]:
def skos_relation(conf):
    if conf >= 0.90: return "skos:exactMatch"
    if conf >= 0.70: return "skos:closeMatch"
    return "skos:relatedMatch"

cands = {}
for key, ev in exact_pairs.items():
    both_pref = (ev["ta"] == "preferred" and ev["tb"] == "preferred")
    conf = 0.75 + 0.20 * ev["w"]      # 0.85 (hidden) .. 0.95 (preferred~preferred)
    cands[key] = {"conf": conf, "method": f"exact_label:{ev['ta']}~{ev['tb']}", "both_pref": both_pref}
for key, s in fuzzy_pairs.items():
    if key in cands:
        cands[key]["conf"] = max(cands[key]["conf"], s)
        cands[key]["method"] += "+tfidf" if not USE_EMBEDDINGS else "+emb"
        cands[key]["fuzzy"] = s
    else:
        cands[key] = {"conf": round(s, 3), "both_pref": False, "fuzzy": s,
                      "method": "tfidf_charngram" if not USE_EMBEDDINGS else "embeddings"}

# Règle anti-inflation : la tranche exactMatch (>=0.90) n'est atteignable que si
# la correspondance repose sur un libellé *préféré~préféré* exact, OU si la
# similarité floue dépasse indépendamment 0.90. Une correspondance basée seulement
# sur un *alias* exact (gonflée par un petit bonus flou) est ramenée à 0.89 (closeMatch).
for key, c in cands.items():
    if c["conf"] >= 0.90 and not (c.get("both_pref") or c.get("fuzzy", 0) >= 0.90):
        c["conf"] = 0.89

align_rows = []
for (a, b), c in cands.items():
    oa, ob = id2occ[a], id2occ[b]
    conf = round(c["conf"], 3)
    validated = "auto" if (conf >= 0.90 and c.get("both_pref")) else "pending"
    align_rows.append({
        "entity_id_a": a, "source_a": oa["source"],
        "entity_id_b": b, "source_b": ob["source"],
        "relation": skos_relation(conf), "confidence": conf, "method": c["method"],
        "validated": validated,
        "notes": f"{oa['pref_label_fr'] or oa['pref_label_en']} <> {ob['pref_label_fr'] or ob['pref_label_en']}",
    })
align_rows.sort(key=lambda r: -r["confidence"])

C.write_csv(C.ALIGNMENTS_CSV, C.ALIGNMENT_FIELDS, align_rows)
print(f"{len(align_rows)} alignements écrits -> concept_alignments.csv")
print("Par relation :", dict(Counter(r["relation"] for r in align_rows)))
print("Par statut   :", dict(Counter(r["validated"] for r in align_rows)))
print("\nMeilleures correspondances :")
for r in align_rows[:10]:
    print(f"  {r['confidence']:.2f} {r['relation']:18s} {r['validated']:7s} | {r['notes'][:58]}")

545 alignements écrits -> concept_alignments.csv
Par relation : {'skos:exactMatch': 7, 'skos:closeMatch': 124, 'skos:relatedMatch': 414}
Par statut   : {'auto': 1, 'pending': 544}

Meilleures correspondances :
  0.98 skos:exactMatch    auto    | Architecte cloud <> architecte cloud
  0.96 skos:exactMatch    pending | Administrateur / Administratrice sécurité informatique <> 
  0.95 skos:exactMatch    pending | développeur web/développeuse web <> Développeur / Développ
  0.93 skos:exactMatch    pending | ingénieur/ingénieure de données <> Data engineer
  0.91 skos:exactMatch    pending | analyste de système informatique <> Analyste d'étude infor
  0.91 skos:exactMatch    pending | technicien informatique/technicienne informatique <> Techn
  0.91 skos:exactMatch    pending | Analyste d'application informatique <> analyste de système
  0.90 skos:closeMatch    pending | Développeur / Développeuse - jeux vidéo <> développeur de 
  0.90 skos:closeMatch    pending | technicien informatique/te

## 6.4. Revue + évaluation

Tous les alignements sont exportés dans un fichier de revue. Annotez chaque alignement (`gold = 1` si correct, `0` si faux). Le notebook mesure ensuite la qualité sur l'ensemble des annotations.

In [14]:
# --- Rapport de qualité AUTONOME ---
def evidence_signals(r):
    m = r["method"]
    pref_exact = "exact_label:preferred~preferred" in m
    corroborated = m.startswith("exact_label") and ("tfidf" in m or "emb" in m)
    return pref_exact, corroborated

n_pref = sum(1 for r in align_rows if evidence_signals(r)[0])
n_corr = sum(1 for r in align_rows if evidence_signals(r)[1])
print("=== Rapport de qualité autonome ===")
print(f"  Total alignements                                   : {len(align_rows)}")
print(f"  Reposant sur libellé préféré~préféré exact (fort)   : {n_pref}")
print(f"  Corroborés par 2 signaux indépendants (exact+flou)  : {n_corr}")
print(f"  Auto-validés (haute confiance, préféré~préféré)     : {sum(1 for r in align_rows if r['validated']=='auto')}")
print(f"  À réviser (pending)                                 : {sum(1 for r in align_rows if r['validated']=='pending')}")
bands = Counter("≥0.90" if float(r["confidence"]) >= 0.9 else
                "0.70–0.90" if float(r["confidence"]) >= 0.7 else "<0.70" for r in align_rows)
print("  Répartition par confiance :", dict(bands))

# --- Export de TOUS les alignements pour revue  ---

REVIEW_CSV = os.path.join(C.CANONICAL_DIR, "alignment_review.csv")

# Si le fichier existe déjà avec des annotations, on les préserve.
existing_gold = {}
if os.path.isfile(REVIEW_CSV):
    with open(REVIEW_CSV, encoding="utf-8-sig") as f:
        for r in csv.DictReader(f):
            key = (r.get("entity_id_a",""), r.get("entity_id_b",""))
            if r.get("gold") in ("0", "1"):
                existing_gold[key] = r["gold"]

review_fields = ["gold", "label_a", "label_b", "relation", "confidence",
                 "source_a", "source_b", "entity_id_a", "entity_id_b", "method"]
with open(REVIEW_CSV, "w", encoding="utf-8-sig", newline="") as f:
    w = csv.DictWriter(f, fieldnames=review_fields)
    w.writeheader()
    for r in align_rows:
        la, lb = r["notes"].split(" <> ", 1) if " <> " in r["notes"] else (r["notes"], "")
        key = (r["entity_id_a"], r["entity_id_b"])
        gold = existing_gold.get(key, "")
        w.writerow({"gold": gold, "label_a": la, "label_b": lb,
                    "relation": r["relation"], "confidence": r["confidence"],
                    "source_a": r["source_a"], "source_b": r["source_b"],
                    "entity_id_a": r["entity_id_a"], "entity_id_b": r["entity_id_b"],
                    "method": r["method"]})

n_already = sum(1 for v in existing_gold.values() if v in ("0","1"))
print(f"\nRevue exportée ({len(align_rows)} paires) : {REVIEW_CSV}")
if n_already:
    print(f"  {n_already} annotation(s) existante(s) préservées.")
print("Ouvrez le fichier dans un tableur. Remplissez la colonne 'gold' :")
print("  1 = alignement correct   |   0 = alignement faux")
print("Sauvegardez, puis exécutez la cellule suivante pour l'évaluation P/R/F1.")


=== Rapport de qualité autonome ===
  Total alignements                                   : 545
  Reposant sur libellé préféré~préféré exact (fort)   : 1
  Corroborés par 2 signaux indépendants (exact+flou)  : 4
  Auto-validés (haute confiance, préféré~préféré)     : 1
  À réviser (pending)                                 : 544
  Répartition par confiance : {'≥0.90': 7, '0.70–0.90': 124, '<0.70': 414}

Revue exportée (545 paires) : d:\JobKB-final\canonical\alignment_review.csv
Ouvrez le fichier dans un tableur. Remplissez la colonne 'gold' :
  1 = alignement correct   |   0 = alignement faux
Sauvegardez, puis exécutez la cellule suivante pour l'évaluation P/R/F1.


In [15]:
# Évaluation des annotations 

REVIEW_CSV = os.path.join(C.CANONICAL_DIR, "alignment_review.csv")

def load_gold():
    if not os.path.isfile(REVIEW_CSV):
        return []
    with open(REVIEW_CSV, encoding="utf-8-sig") as f:
        return [r for r in csv.DictReader(f) if r.get("gold") in ("0", "1")]

def evaluate(labeled, threshold):
    tp = sum(1 for r in labeled if r["gold"] == "1" and float(r["confidence"]) >= threshold)
    fp = sum(1 for r in labeled if r["gold"] == "0" and float(r["confidence"]) >= threshold)
    fn = sum(1 for r in labeled if r["gold"] == "1" and float(r["confidence"]) <  threshold)
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec  = tp / (tp + fn) if tp + fn else 0.0
    f1   = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    return dict(threshold=threshold, tp=tp, fp=fp, fn=fn,
                precision=round(prec, 3), recall=round(rec, 3), f1=round(f1, 3))

labeled = load_gold()
total_align = len(align_rows)
if not labeled:
    print("Aucune annotation 'gold' trouvée pour l'instant.")
    print(f"Ouvrez {REVIEW_CSV} ({total_align} paires), remplissez la colonne 'gold',")
    print("puis réexécutez cette cellule pour obtenir les métriques P/R/F1.")
    print("\nLe rapport de qualité autonome (cellule précédente) fournit déjà des signaux utiles.")
else:
    coverage = len(labeled) / total_align * 100 if total_align else 0
    print(f"{len(labeled)}/{total_align} paires annotées ({coverage:.0f}% de couverture). Balayage de seuil :")
    results = [evaluate(labeled, t) for t in (0.70, 0.75, 0.80, 0.85, 0.90)]
    for r in results:
        print(f"  seuil={r['threshold']:.2f}  P={r['precision']:.2f}  R={r['recall']:.2f}  F1={r['f1']:.2f}  (tp={r['tp']} fp={r['fp']} fn={r['fn']})")
    best = max(results, key=lambda r: r["f1"])
    print(f"\nMeilleur F1 = {best['f1']:.2f} au seuil {best['threshold']:.2f}.")
    if coverage < 100:
        print(f"\nNote : {total_align - len(labeled)} paires non encore annotées. Complétez-les pour une évaluation exhaustive.")


545/545 paires annotées (100% de couverture). Balayage de seuil :
  seuil=0.70  P=0.78  R=0.44  F1=0.56  (tp=102 fp=29 fn=128)
  seuil=0.75  P=0.79  R=0.30  F1=0.44  (tp=70 fp=19 fn=160)
  seuil=0.80  P=0.87  R=0.20  F1=0.33  (tp=46 fp=7 fn=184)
  seuil=0.85  P=0.86  R=0.13  F1=0.23  (tp=30 fp=5 fn=200)
  seuil=0.90  P=1.00  R=0.03  F1=0.06  (tp=7 fp=0 fn=223)

Meilleur F1 = 0.56 au seuil 0.70.
